# 3D Gaussian Splatting (Nerfstudio) 学習ノートブック
ローカルPCでCOLMAP処理済みの `processed_data.zip` をアップロードし、学習して3Dモデル(.ply)を書き出します。

**事前準備**
1. ローカルで `model/process_video.sh input/input.mp4` を実行し、`model/output/processed_data.zip` を作成する
2. Colab左側のファイルパネルを開き、`processed_data.zip` を `/content` 直下にドラッグ&ドロップする

**※重要※**
実行前に、メニューの「ランタイム」>「ランタイムのタイプを変更」から、ハードウェア アクセラレータを「**T4 GPU**」に設定してください。


In [ ]:
# @title 1. 環境構築 (Nerfstudioのインストール)
# 数分かかります。COLMAPはローカルで処理済みのため不要です。
!pip install nerfstudio


In [ ]:
# @title 2. 学習データの展開
# 左側のファイルパネルに置いた processed_data.zip を展開します。
import os

zip_path = '/content/processed_data.zip'
output_dir = '/content/processed_data'

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"{zip_path} が見つかりません。ファイルパネルへのアップロードを確認してください。")

# 前回実行時のデータが残っていれば削除
!rm -rf {output_dir}
!unzip -q "{zip_path}" -d /content
print(f"展開完了: {output_dir}")


In [ ]:
# @title 3. 3D Gaussian Splattingの学習
# GPUメモリ不足回避のためViewerは起動せず、ログはTensorBoardに出力します。学習完了後に自動で終了します。
!ns-train splatfacto --data "{output_dir}" --vis tensorboard


In [ ]:
# @title 4. 3Dモデル(.ply)の書き出しとダウンロード
import glob
import os
from google.colab import files

# 学習結果のconfig.ymlを検索
config_paths = glob.glob('/content/outputs/processed_data/splatfacto/*/config.yml')

if not config_paths:
    print("学習結果が見つかりません。")
else:
    # 最新のconfigを使用
    config_paths.sort(key=os.path.getmtime, reverse=True)
    config_path = config_paths[0]

    export_dir = '/content/exports'
    os.makedirs(export_dir, exist_ok=True)

    print("3Dモデル(.ply)を生成しています...")
    !ns-export gaussian-splat --load-config "{config_path}" --output-dir "{export_dir}"

    ply_files = glob.glob(os.path.join(export_dir, '*.ply'))
    if ply_files:
        # 最新のplyファイルをダウンロード
        ply_files.sort(key=os.path.getmtime, reverse=True)
        ply_file = ply_files[0]
        print(f"ダウンロードを開始します: {ply_file}")
        files.download(ply_file)
    else:
        print("plyファイルの生成に失敗しました。")
